# Session 26: Dask, when the data does not fit

A batch job gets 0.94 GB of memory per CPU unless it asks for more: 3.77 GB for a job with 4 CPUs. Today's data, 100
million taxi trips, take 3.8 GB in pandas, and reading them takes almost three times that. We answer one question
about the trips with pandas and with Dask, and look at how Dask splits the data and the work.

The question: **what is the mean fare for each pickup zone?**

- **Part 1**: pandas on the whole table (I show it in class, you run it at home).
- **Part 2**: the same question in Dask.
- **Part 3**: what Dask builds before it runs anything.
- **Part 4**: chunk size and number of workers (homework).
- **Part 5**: a performance report (homework).

Run the cells in order, from top to bottom: click a cell and press Shift+Enter, on Mac and on Windows. When the text
says **Predict**, decide on your answer before you run the next cell.

## Before you start

Look at the top right corner of this notebook. It must say **OSPC (Python 3.12)**. If it says something else, click
on that name and choose OSPC (Python 3.12) from the list.

The trips are in a shared folder that everybody can read: 100 Parquet files of one million trips each, one year in
time order. The measurements run as batch jobs with 4 CPUs. The next cell puts the course Python in the `PATH` for
the jobs and lists the files.

In [ ]:
import os
import time

os.environ["PATH"] = "/data/software/envs/ospc/bin:" + os.environ["PATH"]
DATA = "/data/ospc/trips"

files = sorted(os.listdir(DATA))
size = sum(os.path.getsize(os.path.join(DATA, f)) for f in files)
print(len(files), "files, from", files[0], "to", files[-1], f"{size / 1e9:.2f} GB on disk")
print("CPUs of your session:", len(os.sched_getaffinity(0)))

## Part 1. pandas on the whole table

The next cell writes `trips.py`. It answers the question in one of four ways and prints the node, the CPUs and the
memory of the job, the time, and the most memory the program used (its peak). The libraries are imported before the
clock starts: importing Dask takes about a second, and it would hide the time of the question.

In [ ]:
%%writefile trips.py
"""Mean fare per pickup zone.

    python trips.py METHOD DATA [WORKERS ...]

METHOD: pandas (all the columns), pandas-cols (only the two columns we need), dask (threads, the default) or
dask-processes (4 worker processes on one node, a LocalCluster; it also writes report.html).
"""
import os
import resource
import sys
import time


def job_memory_gb():
    """The memory Slurm gave this job: the smallest memory.max on the path of its control group."""
    path = open("/proc/self/cgroup").read().strip().split(":")[-1]
    limits = []
    while path not in ("", "/"):
        try:
            value = open(f"/sys/fs/cgroup{path}/memory.max").read().strip()
            if value != "max":
                limits.append(int(value))
        except OSError:
            pass
        path = os.path.dirname(path)
    return min(limits) / 1e9 if limits else float("nan")


def peak_gb():
    """The most memory this process has used so far (Linux gives it in KiB)."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024 / 1e9


def show(name, seconds, memory, result):
    print(f"{name}: {seconds:.2f} s, peak memory {memory:.2f} GB; {len(result)} zones, "
          f"zone 1 mean fare {result.loc[1]:.2f}", flush=True)


def main():
    method, data = sys.argv[1], sys.argv[2]
    cpus = len(os.sched_getaffinity(0))
    workers = [int(w) for w in sys.argv[3:]] or [cpus]
    print(f"node {os.uname().nodename}, {cpus} CPUs, job memory {job_memory_gb():.2f} GB", flush=True)

    t = time.perf_counter()
    import pandas as pd
    import pyarrow.parquet  # noqa: F401
    if method.startswith("dask"):
        import dask
        import dask.dataframe as dd
    print(f"importing the libraries: {time.perf_counter() - t:.1f} s", flush=True)

    if method in ("pandas", "pandas-cols"):
        columns = ["pu_zone", "fare"] if method == "pandas-cols" else None
        t = time.perf_counter()
        df = pd.read_parquet(data, columns=columns)
        result = df.groupby("pu_zone")["fare"].mean()
        seconds = time.perf_counter() - t
        print(f"the table: {len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e9:.2f} GB in pandas")
        show(method, seconds, peak_gb(), result)

    elif method == "dask":
        for n in workers:
            with dask.config.set(scheduler="threads", num_workers=n):
                t = time.perf_counter()
                trips = dd.read_parquet(data)
                result = trips.groupby("pu_zone")["fare"].mean().compute()
                show(f"dask, {n} threads", time.perf_counter() - t, peak_gb(), result)

    elif method == "dask-processes":
        from dask.distributed import Client, LocalCluster, performance_report
        n = workers[0]
        t = time.perf_counter()
        with LocalCluster(n_workers=n, threads_per_worker=1, dashboard_address=":0") as cluster, \
                Client(cluster) as client:
            limit = min(w["memory_limit"] for w in client.scheduler_info()["workers"].values())
            print(f"starting {n} worker processes: {time.perf_counter() - t:.2f} s, "
                  f"memory limit {limit / 1e9:.2f} GB each", flush=True)
            with performance_report(filename="report.html"):
                t = time.perf_counter()
                trips = dd.read_parquet(data)
                result = trips.groupby("pu_zone")["fare"].mean().compute()
                seconds = time.perf_counter() - t
            memory = peak_gb() + sum(client.run(peak_gb).values())
        show(f"dask, {n} processes", seconds, memory, result)
        print("wrote report.html")


if __name__ == "__main__":   # worker processes import this file: only the main process runs main()
    main()

pandas reads the 100 files into one DataFrame of 3.8 GB, then groups it.

**Predict**: will it fit in a job with 4 CPUs and the default memory, 3.77 GB?

I ran it for you on julia. Do not run it in class: when a job needs more memory than it asked for, our cluster does not
stop it at once. Linux moves part of its memory to swap, on disk, and the job slows to a crawl. The job is stopped only
when the node's swap is full, and the swap of a node (8 GB) is shared by every job on it.

| Job with 4 CPUs | Time | Peak memory |
|---|---|---|
| default memory, 3.77 GB | 22.8 s, with 6.4 GB pushed to swap | at the limit |
| `--mem=16G` | 2.34 s | 10.3 GB |
| `--mem=16G`, only the two columns (`pandas-cols`) | 1.27 s | 4.0 GB |

While it reads, pandas needs about 2.7 times the size of the table (10.3 GB for 3.8 GB): the copy that pyarrow reads
from the files and the copy that pandas builds from it exist at the same time.

**At home**, run it with enough memory (about 10 s, most of it waiting for the job to start):

In [ ]:
!sbatch -p cpu -c 4 --mem=16G -t 5 -o pandas.out --wait --wrap "python trips.py pandas {DATA}"
!cat pandas.out

## Part 2. The same question in Dask

Dask reads the folder as a collection of partitions, each a pandas DataFrame, and turns the question into many small
tasks that run on the 4 CPUs of the job. The two lines that matter in `trips.py`:

```python
trips = dd.read_parquet(data)
result = trips.groupby("pu_zone")["fare"].mean().compute()
```

The same pandas calls, and `compute()` at the end returns a pandas result. This job keeps the default memory,
3.77 GB.

**Predict**: faster or slower than pandas with 16 GB (2.34 s)? How much memory will it use?

In [ ]:
!sbatch -p cpu -c 4 -t 5 -o dask.out --wait --wrap "python trips.py dask {DATA}"
!cat dask.out

On julia it took 1.04 s and 1.2 GB, plus about a second to import Dask. Dask is faster than pandas even when pandas
fits: pandas' groupby runs on one CPU, Dask's on all four. Its memory stays at about 1.2 GB whatever the number of
trips, because it never holds the whole table, only a few partitions at a time. And it reads only the two columns
the question uses, without being told.

Your times may differ from mine: the node matters (the first line of the output says which node ran the job).

## Part 3. What Dask builds before it runs

These cells run in your JupyterLab session, not in a job: they build plans and read almost nothing.

`dask.delayed` wraps a function call. Instead of running it, Dask records it as a **task** in a **graph**. Nothing
runs until `compute()`.

In [ ]:
import dask
from dask import delayed


def slow_square(x):
    time.sleep(0.5)
    return x * x


squares = [delayed(slow_square)(i) for i in range(4)]
total = delayed(sum)(squares)
print(total)
print(len(total.__dask_graph__()), "tasks in the graph, and nothing has run yet")

**Predict**: four tasks of 0.5 s each, and your session has 2 CPUs. How long will `compute()` take?

In [ ]:
t = time.perf_counter()
print(total.compute())
print(f"{time.perf_counter() - t:.1f} s")

About 1 s: Dask's threads ran two `slow_square` tasks at a time (sleeping, like reading a file, lets other threads
run), and `sum` ran when its four inputs were ready. It is the same idea as `pool.submit` in Lecture 15, with one
difference: Dask knows the whole graph before it starts, so it can choose the order and free each result as soon as
nobody needs it.

`dask.dataframe` builds the same kind of graph from pandas calls. Up to `compute()` it reads only the files'
metadata:

In [ ]:
import dask.dataframe as dd

trips = dd.read_parquet(DATA)
question = trips.groupby("pu_zone")["fare"].mean()
print(trips.npartitions, "partitions, one per file")
print(trips.columns.tolist())
print("nothing has been read yet")

Before it runs, Dask rewrites the question into a plan. With 100 partitions the plan is **map, shuffle, reduce**:
sums and counts of the fares by zone in each partition (`GroupByChunk`), a shuffle that sends the pieces of each zone
to one place, then the means (`Aggregate`). Look also at the columns it reads.

In [ ]:
question.optimize().pprint()

With few groups (we have 263 zones) a **tree** does the same job: the partial sums and counts are combined a few
at a time until one is left. `split_out=1` asks for it:

In [ ]:
trips.groupby("pu_zone")["fare"].mean(split_out=1).optimize().pprint()

On julia, with 4 threads, both plans took 1.0 s. With 4 worker processes the tree was 11% faster, because the
shuffle moves pieces between processes. The shuffle is what scales when the groups are many, millions of keys,
because no single task then has to hold every group.

## Part 4. Homework: chunk size and workers

Every Dask task costs time to schedule: about 0.2 ms here, against 0.005 ms for a task of a `ThreadPoolExecutor`. So
the size of the pieces matters. The next cell writes `chunks.py`: it sums 100 million random numbers with NumPy, then
with `dask.array` split into chunks of 1,000 to 100 million numbers. One chunk is one task.

**Predict**: which chunk size will be fastest on 4 CPUs? How slow will chunks of 1,000 numbers be?

In [ ]:
%%writefile chunks.py
import os
import time

import dask.array as da
import numpy as np

n = 100_000_000
print(f"node {os.uname().nodename}, {len(os.sched_getaffinity(0))} CPUs", flush=True)
t = time.perf_counter()
np.random.default_rng(1).random(n).sum()
print(f"NumPy, one call              {time.perf_counter() - t:6.2f} s", flush=True)
for chunk in (1_000, 10_000, 100_000, 1_000_000, 10_000_000, 100_000_000):
    x = da.random.default_rng(1).random(n, chunks=chunk)
    t = time.perf_counter()
    x.sum().compute()
    print(f"chunks of {chunk:>11,}: {n // chunk:>7,} tasks {time.perf_counter() - t:6.2f} s", flush=True)

In [ ]:
!sbatch -p cpu -c 4 -t 5 -o chunks.out --wait --wrap "python chunks.py"
!cat chunks.out

Chunks of 1,000 numbers take about 20 s: 100,000 tasks of 0.2 ms each, and almost no work in each. One chunk of 100
million numbers is one task on one thread, like NumPy. The best is in between, 1 million numbers (8 MB) per chunk:
100 tasks, enough to keep 4 CPUs busy, each with about 1 ms of work. Rule of thumb: tasks of 100 ms or more, so that
the overhead stays under 1%.

Now the number of workers. The next job asks for 16 CPUs and answers the question with 1, 2, 4, 8 and 16 threads.

**Predict**: with 16 threads, how many times faster than with 1? Think of Amdahl (Lecture 16): planning the graph and
the last steps do not get faster.

In [ ]:
!sbatch -p cpu -c 16 -t 10 -o workers.out --wait --wrap "python trips.py dask {DATA} 1 2 4 8 16"
!cat workers.out

On julia (in a 64-CPU job): 2.40 s with 1 thread, 0.89 s with 4, 0.52 s with 16, then flat. Amdahl fits with about
0.4 s that does not shrink, so the speedup stops at about 5. The memory grows with the threads, because each one
holds its own partitions.

## Part 5. Homework: a performance report

Dask can also run the tasks in worker processes, with a `LocalCluster`: use it when your tasks run Python code that
holds the GIL. The next job runs the question on 4 worker processes with one thread each, and writes a
**performance report**, `report.html`: what every worker did, and when.

**Predict**: faster or slower than with 4 threads?

In [ ]:
!sbatch -p cpu -c 4 -t 5 -o processes.out --wait --wrap "python trips.py dask-processes {DATA}"
!cat processes.out

On julia: 2.2 s, plus 0.5 s to start the workers, against 1.04 s with threads. Processes must be started, every
piece of the shuffle is copied from one process to another, and each task costs about 1 ms to hand out, against
0.2 ms with threads. Dask gave each worker a quarter of the job's memory (0.94 GB), and it was enough.

Open `report.html`: double-click it in the file browser on the left, then click **Trust HTML** at the top of the tab
(JupyterLab does not run the code of an HTML file until you say you trust it). At the top: the number of tasks
(141 on julia) and the duration. The first tab, Task Stream, has one row per worker: each bar is a task, and the
gaps are time the worker waited. On julia the first second had no tasks at all: Dask was planning the graph and
sending it to the scheduler. Then the 100 files were read by 25 tasks of 4 files each: the question uses only 2 of
the 8 columns, so Dask joins several files into one task.

1. Hover over the bars. Which tasks take most of the time? Find the tasks of the shuffle (their names start with
   `shuffle`): when do they run?
2. In `trips.py`, add `memory_limit="0.85GB"` to `LocalCluster(...)` and run it again. **Predict** first: what will
   the workers do when their data do not fit? Look for warnings in `processes.out` and for red bars (reading from
   disk) in the report.
3. Change `threads_per_worker=1` to `2` and `n_workers=n` to `n // 2`: 2 processes with 2 threads each. Faster or
   slower than 4 processes?